In [1]:
import pandas as pd
import sqlite3
from pathlib import Path

project_root = Path.cwd().parent
db_path = project_root / "data" / "cloudtrail.db"

assert db_path.exists(), "SQLite database not found"

connection = sqlite3.connect(db_path)

In [2]:
pd.read_sql_query(
    "SELECT COUNT(*) AS total_events FROM events",
    connection
)

,total_events
0,95


In [3]:
df_activity = pd.read_sql_query(
    """
    SELECT
        user,
        COUNT(*) AS event_count
    FROM events
    GROUP BY user
    ORDER BY event_count DESC
    """,
    connection
)

df_activity

,user,event_count
0,admin,36
1,developer,22
2,service-user,19
3,analyst,18


In [4]:
df_ranked_users = pd.read_sql_query(
    """
    SELECT
        user,
        COUNT(*) AS event_count,
        RANK() OVER (ORDER BY COUNT(*) DESC) AS activity_rank
    FROM events
    GROUP BY user
    ORDER BY event_count DESC
    """,
    connection
)

df_ranked_users

,user,event_count,activity_rank
0,admin,36,1
1,developer,22,2
2,service-user,19,3
3,analyst,18,4


In [5]:
pd.read_sql_query(
    """
    WITH ranked_users AS (
        SELECT
            user,
            COUNT(*) AS event_count,
            RANK() OVER (ORDER BY COUNT(*) DESC) AS activity_rank
        FROM events
        GROUP BY user
)
    SELECT *
    FROM ranked_users
    WHERE activity_rank <= 3
    ORDER BY activity_rank;""",
    connection)

,user,event_count,activity_rank
0,admin,36,1
1,developer,22,2
2,service-user,19,3


In [6]:
df_daily = pd.read_sql_query(
    """
    SELECT
        DATE(timestamp) AS event_date,
        COUNT(*) AS event_count,
        LAG(COUNT(*)) OVER (
            ORDER BY DATE(timestamp)) AS prev_day_event_count,
        COUNT(*) - LAG(COUNT(*)) OVER (
                    ORDER BY DATE(timestamp)) AS event_count_change
    FROM events
    GROUP BY DATE(timestamp)
    ORDER BY event_date
    """,
    connection
)

df_daily

,event_date,event_count,prev_day_event_count,event_count_change
0,2026-11-10,17,NaN,NaN
1,2026-11-11,19,17.0,2.0
2,2026-11-12,17,19.0,-2.0
3,2026-11-13,17,17.0,0.0
4,2026-11-14,25,17.0,8.0


In [7]:
# Could improve readability of df_daily query by using a CTE first for the aggregations

df_daily_improved = pd.read_sql_query(
    """
    WITH daily_counts AS(
        SELECT
            DATE(timestamp) AS event_date,
            COUNT(*) AS event_count
        FROM events
        GROUP BY DATE(timestamp)
        )

        SELECT
            event_date,
            event_count,
            LAG(event_count) OVER (
                ORDER BY event_date) AS prev_day_count,
            event_count - LAG(event_count) OVER (
                ORDER BY event_date) AS event_count_change
        FROM daily_counts
        ORDER BY event_date
    """,
    connection
)

df_daily_improved

,event_date,event_count,prev_day_count,event_count_change
0,2026-11-10,17,NaN,NaN
1,2026-11-11,19,17.0,2.0
2,2026-11-12,17,19.0,-2.0
3,2026-11-13,17,17.0,0.0
4,2026-11-14,25,17.0,8.0


In [8]:
assert df_daily_improved["event_count"].sum() == 95

In [9]:
pd.testing.assert_frame_equal(
    df_daily[
        ["event_date", "event_count", "event_count_change"]
    ].reset_index(drop=True),
    df_daily_improved[
        ["event_date", "event_count", "event_count_change"]
    ].reset_index(drop=True)
)

**Missing-Date Test — Understanding LAG()**

This hypothetical example demonstrates how `LAG()` behaves when calendar dates are missing from a dataset. I learned that `LAG()` retrieves the previous available row rather than necessarily the previous calendar day's value.

For example, if November 14 is missing, November 15 will be compared with November 13. This could produce a misleading day-over-day change.

Before interpreting the results, I would investigate whether November 14 genuinely had zero events or whether its data was not collected. A complete calendar table or a consecutive-date validation could help address this limitation.

In [10]:
df_missing_days = pd.DataFrame({
    "event_date": [
        "2026-11-12",
        "2026-11-13",
        "2026-11-15"
    ],
    "event_count": [10, 15, 8]
})

df_missing_days.to_sql(
    "daily_activity_test",
    connection,
    if_exists="replace",
    index=False
)

3

In [11]:
pd.read_sql_query(
    """
    SELECT
        event_date,
        event_count,
        LAG(event_count) OVER (
            ORDER BY event_date) AS prev_count
    FROM daily_activity_test
    ORDER BY event_date
    """,
    connection
)

,event_date,event_count,prev_count
0,2026-11-12,10,NaN
1,2026-11-13,15,10.0
2,2026-11-15,8,15.0


In [12]:
df_rank_test = pd.DataFrame({
    "user": ["admin", "developer", "analyst", "service-user"],
    "event_count": [36, 22, 22, 15]
})

df_rank_test.to_sql(
    "ranking_test",
    connection,
    if_exists="replace",
    index=False
)

4

In [13]:
df_rank_results = pd.read_sql_query(
    """
    SELECT
        user,
        event_count,
        ROW_NUMBER() OVER (
            ORDER BY event_count DESC, user ASC
        ) AS row_num,
        RANK() OVER (
            ORDER BY event_count DESC
        ) AS activity_rank,
        DENSE_RANK() OVER (
            ORDER BY event_count DESC
        ) AS dense_rank
    FROM ranking_test
    ORDER BY event_count DESC, user ASC;
    """,
    connection
)

df_rank_results

,user,event_count,row_num,activity_rank,dense_rank
0,admin,36,1,1,1
1,analyst,22,2,2,2
2,developer,22,3,2,2
3,service-user,15,4,4,3


In [14]:
service_user = df_rank_results.loc[
    df_rank_results["user"] == "service-user"
].iloc[0]

assert service_user["row_num"] == 4
assert service_user["activity_rank"] == 4
assert service_user["dense_rank"] == 3

### Day 19 — SQL Window Functions: Ranking and Comparing Activity

Practiced SQL window functions using the Cloud Security Intelligence Platform's 95 deduplicated synthetic CloudTrail events.

**Key accomplishments:**
- Used `RANK()` to rank identities by total CloudTrail event activity.
- Created a CTE to identify the top three identities.
- Used `LAG()` to compare daily event counts with the preceding available date.
- Calculated changes in event volume between consecutive rows.
- Refactored the daily activity query into a CTE for improved readability and maintainability.
- Tested ranking ties using `ROW_NUMBER()`, `RANK()`, and `DENSE_RANK()`.
- Explored the difference between missing dates, zero activity, and unknown activity.

**Results:**
- Admin generated the most events (36), followed by developer (22), service-user (19), and analyst (18).
- Verified that the daily aggregation totals 95 events and that the refactored query matches the original results.
- Demonstrated how ranking functions handle tied values differently.

**Key takeaway:** Window functions allow us to rank and compare observations without losing the detail needed for analysis. However, tie behavior and missing dates must be considered carefully to avoid misleading conclusions.

**Limitation:** The dataset is synthetic. Additionally, `LAG()` references the previous available row, which may not represent the previous calendar day when dates are missing.